In [1]:
import pandas as pd 


data = pd.read_csv("UL_PRB_data_set.csv")
unseen = pd.read_csv("selected_cells_unseen.csv")

data.head()

,Short name,Date,N.PRB.UL.DrbUsed.Avg[%],N.ThpVol.UL,N.User.RRCConn.Active.UL.Avg
0,cell000,2025-01-20 00:00:00,0.178846,6173.480,0.052
1,cell000,2025-01-20 00:15:00,0.096154,2877.464,0.018
2,cell000,2025-01-20 00:30:00,0.075000,3541.224,0.021
3,cell000,2025-01-20 00:45:00,0.148077,9373.480,0.033
4,cell000,2025-01-20 01:00:00,0.563462,39179.448,0.124


In [2]:
unseen.head()

,Short name,Date,N.PRB.UL.DrbUsed.Avg[%],N.ThpVol.UL,N.User.RRCConn.Active.UL.Avg
0,cell230,2025-02-14 00:00:00,0.315385,16190.448,0.081
1,cell230,2025-02-14 00:15:00,0.001923,11.552,0.000
2,cell230,2025-02-14 00:30:00,0.000000,0.040,0.000
3,cell230,2025-02-14 00:45:00,0.003846,103.104,0.002
4,cell230,2025-02-14 01:00:00,0.000000,0.120,0.000


In [1]:
import os
import numpy as np
import pandas as pd
import pickle

# ==========================================================
# CONFIG
# ==========================================================
PREVIOUS_OUTPUT = "output/C1_quantile_forecasting"
PREPROCESS_DIR = os.path.join(PREVIOUS_OUTPUT, "preprocess")

# ==========================================================
# LOAD
# ==========================================================
cluster_path = os.path.join(PREPROCESS_DIR, "cluster_ids.npy")

print("Loading:", cluster_path)

cluster_ids = np.load(cluster_path)

print("\n===== BASIC CHECK =====")
print("Total samples:", len(cluster_ids))
print("dtype:", cluster_ids.dtype)
print("shape:", cluster_ids.shape)

# ==========================================================
# CLUSTER COUNTS
# ==========================================================
unique, counts = np.unique(cluster_ids, return_counts=True)

print("\n===== CLUSTER DISTRIBUTION =====")
for c, n in zip(unique, counts):
    print(f"Cluster {c}: {n} samples ({n/len(cluster_ids)*100:.2f}%)")

print("\nNumber of clusters found:", len(unique))

# ==========================================================
# EXPECTED CHECK
# ==========================================================
EXPECTED = 6

if len(unique) != EXPECTED:
    print("\nWARNING:")
    print(f"Expected {EXPECTED} clusters but found {len(unique)}")
    
    missing = set(range(EXPECTED)) - set(unique)

    if missing:
        print("Missing cluster labels:", missing)

# ==========================================================
# CHECK IF LABELS ARE STRANGE
# ==========================================================
print("\n===== LABEL VALUES =====")
print(unique)

# ==========================================================
# CHECK OTHER PREPROCESS FILES
# ==========================================================
print("\n===== FILE CHECK =====")

for f in os.listdir(PREPROCESS_DIR):
    print(f)

# ==========================================================
# CHECK IF SAMPLES WERE LOST
# ==========================================================
try:
    X = np.load(
        os.path.join(PREPROCESS_DIR, "X.npy"),
        mmap_mode="r"
    )

    y = np.load(
        os.path.join(PREPROCESS_DIR, "y.npy"),
        mmap_mode="r"
    )

    print("\n===== ARRAY CONSISTENCY =====")
    print("X samples:", len(X))
    print("y samples:", len(y))
    print("cluster samples:", len(cluster_ids))

    if len(X) != len(cluster_ids):
        print("ERROR: X and cluster_ids lengths do not match!")

except Exception as e:
    print("Could not load X/y:", e)


# ==========================================================
# OPTIONAL: SAVE CLUSTER SUMMARY
# ==========================================================
summary = pd.DataFrame({
    "cluster": unique,
    "samples": counts,
    "percentage": counts / len(cluster_ids) * 100
})

summary.to_csv("cluster_debug_summary.csv", index=False)

print("\nSaved cluster_debug_summary.csv")

Loading: output/C1_quantile_forecasting/preprocess/cluster_ids.npy

===== BASIC CHECK =====
Total samples: 1063104
dtype: int32
shape: (1063104,)

===== CLUSTER DISTRIBUTION =====
Cluster -1: 286944 samples (26.99%)
Cluster 0: 489216 samples (46.02%)
Cluster 1: 286944 samples (26.99%)

Number of clusters found: 3

Expected 6 clusters but found 3
Missing cluster labels: {2, 3, 4, 5}

===== LABEL VALUES =====
[-1  0  1]

===== FILE CHECK =====
y.npy
metadata.pkl
cluster_ids.npy
X.npy
scaler.pkl
feature_columns.pkl

===== ARRAY CONSISTENCY =====
X samples: 1063104
y samples: 1063104
cluster samples: 1063104

Saved cluster_debug_summary.csv


In [4]:
import pandas as pd
import glob

files = glob.glob(
    "output/B_cluster/**/*.csv",
    recursive=True
)

print("CSV files found:")
for f in files:
    print(f)

for f in files:
    if "cluster" in f.lower() or "assign" in f.lower():
        print("\nChecking:", f)

        df = pd.read_csv(f)

        print(df.shape)
        print(df.head())

        for c in df.columns:
            if "cluster" in c.lower() or "label" in c.lower():
                print("\n", c)
                print(df[c].value_counts())

CSV files found:
output/B_cluster/tables/statistics.csv
output/B_cluster/tables/cluster_assignments.csv
output/B_cluster/tables/feature_importance.csv
output/B_cluster/tables/cluster_profiles.csv

Checking: output/B_cluster/tables/statistics.csv
(18, 3)
         feature        anova_p     kruskal_p
0          n_obs            NaN           NaN
1   missing_rate            NaN           NaN
2  zero_fraction  6.549952e-127  1.927722e-42
3           mean   2.613191e-48  1.048825e-39
4            std   8.693864e-47  1.459292e-38

Checking: output/B_cluster/tables/cluster_assignments.csv
(226, 2)
  series_id  cluster
0   cell000        0
1   cell001        1
2   cell002        0
3   cell003        1
4   cell004        2

 cluster
cluster
0    61
2    57
4    42
3    30
5    19
1    17
Name: count, dtype: int64

Checking: output/B_cluster/tables/feature_importance.csv
(108, 3)
   cluster        feature   z_score
0        0          n_obs       NaN
1        0   missing_rate       NaN
2        